In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import langchain
import os

In [3]:
### CSV PROCESSING

from langchain_community.document_loaders import CSVLoader
from langchain_community.document_loaders import UnstructuredCSVLoader


C:\Users\raavi\AppData\Local\Temp\ipykernel_3432\655069899.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader


In [4]:
### Method 1: CSVLoader
print("Using CSVLoader:- Row-wise loading")

csv_loader = CSVLoader(
    file_path="data/structured_files/products.csv",
    encoding="utf-8",
    csv_args={
        "delimiter": ",", 
        "quotechar": '"'}
)
csv_docs = csv_loader.load()
print(f"CSVLoader extracted {len(csv_docs)} document(s).")
print(f"First document content (truncated): {csv_docs[0].page_content[:200]}...")
print(f"metadata: {csv_docs[0].metadata}")


Using CSVLoader:- Row-wise loading
CSVLoader extracted 5 document(s).
First document content (truncated): Product: Laptop
Category: Electronics
Price: 999.99
Stock: 50
Description: High-performance laptop with 16GB RAM and 512GB SSD...
metadata: {'source': 'data/structured_files/products.csv', 'row': 0}


In [12]:
from typing import List
import pandas as pd
from langchain_core.documents import Document

def custom_csv_loader(file_path: str) -> List[Document]:
    df = pd.read_csv(file_path)

    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

    print("CSV columns:", df.columns.tolist())

    documents = []

    for idx, row in df.iterrows():
        content = f"""Product Information:
- Product ID: {row.get('product_id', '')}
- Name: {row.get('name', '')}
- Description: {row.get('description', '')}
- Price: ${row.get('price', '')}
- Category: {row.get('category', '')}
- Stock: {row.get('stock', '')}
"""

        doc = Document(
            page_content=content,
            metadata={
                "source": file_path,
                "row_index": idx,
                "product_name": row.get("name", ""),
                "category": row.get("category", ""),
                "price": row.get("price", ""),
            },
        )

        documents.append(doc)

    return documents

In [13]:
custom_csv_loader("data/structured_files/products.csv")

CSV columns: ['product', 'category', 'price', 'stock', 'description']


[Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 0, 'product_name': '', 'category': 'Electronics', 'price': 999.99}, page_content='Product Information:\n- Product ID: \n- Name: \n- Description: High-performance laptop with 16GB RAM and 512GB SSD\n- Price: $999.99\n- Category: Electronics\n- Stock: 50\n'),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 1, 'product_name': '', 'category': 'Accessories', 'price': 29.99}, page_content='Product Information:\n- Product ID: \n- Name: \n- Description: Wireless optical mouse with ergonomic design\n- Price: $29.99\n- Category: Accessories\n- Stock: 200\n'),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 2, 'product_name': '', 'category': 'Accessories', 'price': 79.99}, page_content='Product Information:\n- Product ID: \n- Name: \n- Description: Mechanical keyboard with RGB backlighting\n- Price: $79.99\n- Category: Accessories\n- Stock: 150\n'),
 D

In [19]:
def process_excel_file(file_path: str) -> List[Document]:
    documents = []
    xls = pd.ExcelFile(file_path)

    for sheet_name in xls.sheet_names:
        df = pd.read_excel(xls, sheet_name=sheet_name)

        sheet_content = f"""
Sheet: {sheet_name}

{df.to_string(index=False)}

Columns: {df.columns.tolist()}
Data types: {df.dtypes.astype(str).to_dict()}
Missing values: {df.isnull().sum().to_dict()}
Row count: {len(df)}
"""

        doc = Document(
            page_content=sheet_content,
            metadata={
                "source": file_path,
                "sheet_name": sheet_name,
                "row_count": len(df),
                "columns": df.columns.tolist(),
                "data_types": df.dtypes.astype(str).to_dict(),
                "missing_values": df.isnull().sum().to_dict(),
            },
        )

        documents.append(doc)

    return documents

In [20]:
excel_docs = process_excel_file("data/structured_files/inventory.xlsx")

print(f"Processed {len(excel_docs)} sheets from Excel file.")
print(excel_docs[0].page_content[:500])
print(excel_docs[0].metadata)

Processed 2 sheets from Excel file.

Sheet: Products

 Product    Category  Price  Stock                                         Description
  Laptop Electronics 999.99     50 High-performance laptop with 16GB RAM and 512GB SSD
   Mouse Accessories  29.99    200        Wireless optical mouse with ergonomic design
Keyboard Accessories  79.99    150           Mechanical keyboard with RGB backlighting
 Monitor Electronics 299.99     75                 27-inch 4K monitor with HDR support
  Webcam Electronics  89.99    100             
{'source': 'data/structured_files/inventory.xlsx', 'sheet_name': 'Products', 'row_count': 5, 'columns': ['Product', 'Category', 'Price', 'Stock', 'Description'], 'data_types': {'Product': 'object', 'Category': 'object', 'Price': 'float64', 'Stock': 'int64', 'Description': 'object'}, 'missing_values': {'Product': 0, 'Category': 0, 'Price': 0, 'Stock': 0, 'Description': 0}}


In [23]:
pip install msoffcrypto-tool

Note: you may need to restart the kernel to use updated packages.


In [24]:
from langchain_community.document_loaders import UnstructuredExcelLoader

try:
    excel_loader = UnstructuredExcelLoader("data/structured_files/inventory.xlsx", mode="elements")
    excel_docs = excel_loader.load()
    
except Exception as e:
    print(f"Error using UnstructuredExcelLoader: {e}")


In [28]:
loader = UnstructuredExcelLoader("data/structured_files/inventory.xlsx")
unstructured_docs = loader.load()
unstructured_docs[0].page_content[:500]

'Product Category Price Stock Description Laptop Electronics 999.99 50 High-performance laptop with 16GB RAM and 512GB SSD Mouse Accessories 29.99 200 Wireless optical mouse with ergonomic design Keyboard Accessories 79.99 150 Mechanical keyboard with RGB backlighting Monitor Electronics 299.99 75 27-inch 4K monitor with HDR support Webcam Electronics 89.99 100 1080p webcam with noise cancellation\n\nCategory Total_Items Total_Value Electronics 3 1389.97 Accessories 2 109.98'